In [7]:
# Import Required Libraries

import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# Load the Datasets

# Read both datasets
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

# Display first five records
print("Movies Dataset")
display(movies.head())

print("Credits Dataset")
display(credits.head())

# Display dataset shape
print("Movies Dataset Shape:", movies.shape)
print("Credits Dataset Shape:", credits.shape)

Movies Dataset


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


Credits Dataset


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


Movies Dataset Shape: (4803, 20)
Credits Dataset Shape: (4803, 4)


In [9]:
# Merge the Movies and Credits Datasets

# Merge both datasets using the common 'title' column
movies = movies.merge(credits, on='title')

# Select only the required columns for building the recommendation system
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]

# Check for missing values in each column
print("Missing Values:\n")
print(movies.isnull().sum())

# Display the updated dataset
display(movies.head())

# Display the updated dataset shape
print("\nUpdated Dataset Shape:", movies.shape)

Missing Values:

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."



Updated Dataset Shape: (4809, 7)


In [10]:
# Handle Missing Values

# Check the number of missing values before cleaning
print("Missing Values Before Cleaning:\n")
print(movies.isnull().sum())

# Remove rows containing missing values
movies.dropna(inplace=True)

# Check missing values after cleaning
print("\nMissing Values After Cleaning:\n")
print(movies.isnull().sum())

# Display the updated dataset shape
print("\nDataset Shape After Removing Missing Values:", movies.shape)

Missing Values Before Cleaning:

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

Missing Values After Cleaning:

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

Dataset Shape After Removing Missing Values: (4806, 7)


In [11]:
# Convert String Data into Python Objects

# Import ast library
import ast

# Function to extract the 'name' from each dictionary
def convert(text):
    L = []

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [12]:
# Convert Genres and Keywords into Lists

# Extract genre names
movies['genres'] = movies['genres'].apply(convert)

# Extract keyword names
movies['keywords'] = movies['keywords'].apply(convert)

# Display the updated dataset
movies[['title', 'genres', 'keywords']].head()

,title,genres,keywords
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi..."
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i..."
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel..."


In [13]:
# Extract the Top 3 Cast Members

# Function to extract the names of the first 3 cast members
def convert3(text):
    L = []
    counter = 0

    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

# Apply the function to the cast column
movies['cast'] = movies['cast'].apply(convert3)

# Display the updated cast column
movies[['title', 'cast']].head()

,title,cast
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]"
3,The Dark Knight Rises,"[Christian Bale, Michael Caine, Gary Oldman]"
4,John Carter,"[Taylor Kitsch, Lynn Collins, Samantha Morton]"


In [14]:
# Extract the Director Name

# Function to extract the director from the crew column
def fetch_director(text):
    L = []

    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

# Apply the function to the crew column
movies['crew'] = movies['crew'].apply(fetch_director)

# Display the updated crew column
movies[['title', 'crew']].head()

,title,crew
0,Avatar,[James Cameron]
1,Pirates of the Caribbean: At World's End,[Gore Verbinski]
2,Spectre,[Sam Mendes]
3,The Dark Knight Rises,[Christopher Nolan]
4,John Carter,[Andrew Stanton]


In [15]:
# Convert the Overview into a List of Words

# Split each overview into individual words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Display the updated overview column
movies[['title', 'overview']].head()

,title,overview
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [16]:
# Check the first row of all important columns

print(movies['overview'].iloc[0])
print(movies['genres'].iloc[0])
print(movies['keywords'].iloc[0])
print(movies['cast'].iloc[0])
print(movies['crew'].iloc[0])

['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is', 'dispatched', 'to', 'the', 'moon', 'Pandora', 'on', 'a', 'unique', 'mission,', 'but', 'becomes', 'torn', 'between', 'following', 'orders', 'and', 'protecting', 'an', 'alien', 'civilization.']
['Action', 'Adventure', 'Fantasy', 'Science Fiction']
['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']
['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
['James Cameron']


In [17]:
# Create the Tags Column

# Combine all important features into a single column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Display the title and tags columns
movies[['title', 'tags']].head()

,title,tags
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [18]:
# Create the Final Dataset

# Select only the required columns
new_df = movies[['movie_id', 'title', 'tags']].copy()

# Display the first five records
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [19]:
# Convert Tags from List to String

# Join all words into a single string
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

# Display the updated dataset
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [20]:
# Convert All Text to Lowercase

# Convert all text to lowercase
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

# Display the updated dataset
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [21]:
# Display Information About the Final Dataset

new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4806 entries, 0 to 4808
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4806 non-null   int64 
 1   title     4806 non-null   object
 2   tags      4806 non-null   object
dtypes: int64(1), object(2)
memory usage: 150.2+ KB


In [22]:
# Import Porter Stemmer

from nltk.stem.porter import PorterStemmer

# Create a PorterStemmer object
ps = PorterStemmer()

# Function to perform stemming
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

# Apply stemming to the tags column
new_df['tags'] = new_df['tags'].apply(stem)

# Display the updated dataset
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


In [23]:
# Convert Text into Numerical Vectors

# Import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Create a CountVectorizer object
cv = CountVectorizer(max_features=5000, stop_words='english')

# Convert tags into vectors
vectors = cv.fit_transform(new_df['tags']).toarray()

# Display the shape of the vector matrix
print(vectors.shape)

(4806, 5000)


In [24]:
# Calculate Cosine Similarity

# Import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all movies
similarity = cosine_similarity(vectors)

# Display the shape of the similarity matrix
print(similarity.shape)

(4806, 4806)


In [25]:
# Create a Movie Recommendation Function

def recommend(movie):
    
    # Find the index of the selected movie
    movie_index = new_df[new_df['title'] == movie].index[0]
    
    # Get the similarity scores of the selected movie
    distances = similarity[movie_index]
    
    # Sort the movies based on similarity score
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])
    
    # Display the top 5 recommended movies
    print("Movies Recommended for", movie, ":\n")
    
    for i in movies_list[1:6]:
        print(new_df.iloc[i[0]].title)

In [26]:
# Test the Recommendation System

recommend('Avatar')

Movies Recommended for Avatar :

Aliens
Silent Running
Moonraker
Alien
Mission to Mars


In [27]:
# Test the Recommendation System with Different Movies

print("Recommendations for Avatar")
recommend("Avatar")

print("\nRecommendations for Batman Begins")
recommend("Batman Begins")

print("\nRecommendations for Titanic")
recommend("Titanic")

print("\nRecommendations for Iron Man")
recommend("Iron Man")

Recommendations for Avatar
Movies Recommended for Avatar :

Aliens
Silent Running
Moonraker
Alien
Mission to Mars

Recommendations for Batman Begins
Movies Recommended for Batman Begins :

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin

Recommendations for Titanic
Movies Recommended for Titanic :

The Notebook
Romance & Cigarettes
Veer-Zaara
Poseidon
Ghost Ship

Recommendations for Iron Man
Movies Recommended for Iron Man :

Iron Man 3
Iron Man 2
Avengers: Age of Ultron
Captain America: Civil War
The Avengers


In [28]:
# Save the Final Dataset and Similarity Matrix

import pickle

# Save the processed movie dataset
pickle.dump(new_df, open('movies.pkl', 'wb'))

# Save the similarity matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Files saved successfully!")

Files saved successfully!
